# Compare HSA Delineations

Run spatial method comparisons for a selected network and mode(s), then load the comparison table.


In [ ]:
# ── BOUNDARY VERSION SELECTOR ──────────────────────────────────────────────
# Choose which HSA boundary bundle to use for this analysis.
# Must match a bundle produced by HSA_FINAL.ipynb.
#   'v6'  — original greedy algorithm (no post-selection corrections)
#   'v7'  — + anchor upgrade/demotion + major-orphan promotion
#   'v8'  — + satellite bubble boundaries
BOUNDARY_VERSION = "v7"   # change as needed
# ────────────────────────────────────────────────────────────────────────────


In [ ]:
# Parameters
NETWORK = 'INF'  # e.g., INF, NCD, SYNINF, SYNNCD
HSA_MODE = 'footprint'  # e.g., fewest, footprint, distance, governorate_tau_coverage, governorate_fewest

import os
PIPELINE_VERSION = os.environ.get('PIPELINE_VERSION', 'v7')
OUT_DIR = os.environ.get('HSA_OUT_DIR', os.environ.get('PIPELINE_OUT_DIR', f'out_{PIPELINE_VERSION}'))
DATA_DIR = 'data'
RESULTS_FILE = f'{NETWORK}_spatial_methods_comparison.csv'

POP_RASTER = f'{DATA_DIR}/jor_ppp_2020_UNadj.tif'
HSAS_PATH = f'{OUT_DIR}/{NETWORK}_{HSA_MODE}_hsas_{BOUNDARY_VERSION}.geojson'
FACILITIES_CSV = f'{DATA_DIR}/{NETWORK}_facility_coordinates.csv'
GOVERNORATES_PATH = f'{DATA_DIR}/jordan_governorates.gpkg'
BOUNDARY_PATH = f'{DATA_DIR}/jordan_boundary.gpkg'
OUTPUT_CSV = f'{OUT_DIR}/{NETWORK}_spatial_methods_comparison.csv'

print(f'Network: {NETWORK}')
print(f'HSA mode: {HSA_MODE}')

print(f'HSAs: {HSAS_PATH}')
print(f'Facilities: {FACILITIES_CSV}')


In [ ]:
# Run compare_spatial_methods_v2
import os
import sys
import subprocess

cmd = [
    sys.executable, 'compare_spatial_methods_v2.py',
    '--network', NETWORK,
    '--hsa-mode', HSA_MODE,
    '--population-raster', POP_RASTER,
    '--hsas-path', HSAS_PATH,
    '--facilities-csv', FACILITIES_CSV,
    '--governorates-path', GOVERNORATES_PATH,
    '--boundary-path', BOUNDARY_PATH,
    '--output-csv', OUTPUT_CSV,
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(f'compare_spatial_methods_v2.py failed with code {result.returncode}')


In [ ]:
# Load comparison table
import pandas as pd
from pathlib import Path

results_path = Path(OUTPUT_CSV)
if not results_path.exists():
    raise FileNotFoundError(f'Comparison table not found: {results_path}')

df = pd.read_csv(results_path)
display(df)


In [ ]:
# AUTO_NOTEBOOK_SUMMARY_V1
from pathlib import Path
from datetime import datetime
import os
import json

NOTEBOOK_NAME = "compare_delineations"
NETWORK = globals().get('NETWORK', os.environ.get('NETWORK', 'INF'))
HSA_MODE = globals().get('HSA_MODE', os.environ.get('HSA_MODE', ''))

suffix = f"{NETWORK}_{HSA_MODE}" if HSA_MODE else f"{NETWORK}"

out_root = Path(globals().get('OUT_DIR', globals().get('OUT_ROOT', os.environ.get('HSA_OUT_DIR', os.environ.get('PIPELINE_OUT_DIR', f"out_{os.environ.get('PIPELINE_VERSION', 'v7')}")))))
summary_dir = out_root / 'textresults'
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = summary_dir / f"{NOTEBOOK_NAME}_{suffix}_results.md"

files = [p for p in out_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.md', '.txt', '.png', '.pdf', '.geojson', '.parquet'}]
files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
important = files[:120]

NL = "\n"

blocks = []
blocks.append(f"# Notebook Results: {NOTEBOOK_NAME}")

meta = [
    f"- Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"- Network: {NETWORK}",
]
if HSA_MODE:
    meta.append(f"- HSA mode: {HSA_MODE}")
blocks.append(NL.join(meta))

file_lines = ['## Important Output Files', '']
for p in important:
    file_lines.append(f"- `{p}`")
blocks.append(NL.join(file_lines))

nb_path = Path(f"{NOTEBOOK_NAME}.ipynb")
if nb_path.exists():
    try:
        nb_data = json.loads(nb_path.read_text())
        blocks.append('## Displayed Cell Outputs')

        for idx, cell in enumerate(nb_data.get('cells', []), start=1):
            if cell.get('cell_type') != 'code':
                continue
            outputs = cell.get('outputs', []) or []
            if not outputs:
                continue

            blocks.append(f"### Cell {idx}")

            for out in outputs:
                otype = out.get('output_type')

                if otype == 'stream':
                    text = ''.join(out.get('text', [])) if isinstance(out.get('text', []), list) else str(out.get('text', ''))
                    text = text.rstrip()
                    if text:
                        blocks.append("```text" + NL + text + NL + "```")

                elif otype in ('execute_result', 'display_data'):
                    data = out.get('data', {})
                    if 'text/markdown' in data:
                        md = ''.join(data['text/markdown']) if isinstance(data['text/markdown'], list) else str(data['text/markdown'])
                        md = md.rstrip()
                        if md:
                            blocks.append(md)
                    elif 'text/plain' in data:
                        txt = ''.join(data['text/plain']) if isinstance(data['text/plain'], list) else str(data['text/plain'])
                        txt = txt.rstrip()
                        if txt:
                            blocks.append("```text" + NL + txt + NL + "```")
                    elif 'text/html' in data:
                        html = ''.join(data['text/html']) if isinstance(data['text/html'], list) else str(data['text/html'])
                        html = html.rstrip()
                        if html:
                            blocks.append("```html" + NL + html + NL + "```")
                    elif 'image/png' in data or 'image/jpeg' in data:
                        blocks.append('_[Image output omitted in text summary]_')

                elif otype == 'error':
                    tb = out.get('traceback', []) or []
                    if tb:
                        err = NL.join(str(t) for t in tb)
                    else:
                        err = f"{out.get('ename', 'Error')}: {out.get('evalue', '')}"
                    blocks.append("```text" + NL + err + NL + "```")

    except Exception as e:
        blocks.append('## Displayed Cell Outputs')
        blocks.append(f"Could not parse notebook outputs: {e}")

summary = (NL + NL).join(b for b in blocks if b and str(b).strip()) + NL
summary_path.write_text(summary)
print(f"Saved notebook summary: {summary_path}")
